In [24]:
import os
import pandas as pd
import numpy as np
import json

from model_fitting import (
    build_linear_regression,
    build_random_forest,
    build_xgb,
    train_lstm
)
from cross_validation import (
    ts_cross_val_sklearn,
    ts_cross_val_lstm
)


In [25]:
# Univariate Training set

df_uni = pd.read_csv("/Users/Owner/cmse492_project/data/processed/Univariate_data/Univariate_train_U.csv")

# Multivariate Training Set  
df_multi = pd.read_csv("/Users/Owner/cmse492_project/data/processed/Multivariate_data/Multivariate_train_M.csv")

In [26]:
for df in [df_uni, df_multi]:
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"])
        df.sort_values("datetime", inplace=True)
        df.reset_index(drop=True, inplace=True)

In [27]:
lr_grid = {}  # Linear Regression
rf_grid = {"n_estimators": [300, 400], "max_depth": [10, 20, None]}
xgb_grid = {
    "n_estimators": [300, 400],
    "learning_rate": [0.05, 0.1],
    "max_depth": [6, 8],
    "subsample": [0.8, 0.9],
    "colsample_bytree": [0.8, 0.9]
}
lstm_grid = {"lr": [0.001, 0.0005], "units": [32, 64]}

# Target variable
target = "SO2"



In [28]:
# ----------------------------
# 4. Run CV for UNIVARIATE
# ----------------------------
print("=== UNIVARIATE CV ===")

print("Linear Regression...")
best_lr_uni, lr_results_uni = ts_cross_val_sklearn(build_linear_regression, lr_grid, df_uni, target=target)

#print("Random Forest...")
#best_rf_uni, rf_results_uni = ts_cross_val_sklearn(build_random_forest, rf_grid, df_uni, target=target)

#print("XGBoost...")
#best_xgb_uni, xgb_results_uni = ts_cross_val_sklearn(build_xgb, xgb_grid, df_uni, target=target)



=== UNIVARIATE CV ===
Linear Regression...


In [29]:
print(best_lr_uni)

{'params': {}, 'scores': [0.07320115474103633, 0.4679296087640401, 0.02828205546251993, 0.13578161476781653, 0.06568880967309754], 'mean_mse': np.float64(0.15417664868170206)}


In [30]:
print("Random Forest...")
best_rf_uni, rf_results_uni = ts_cross_val_sklearn(build_random_forest, rf_grid, df_uni, target=target)
print(best_rf_uni)

Random Forest...
{'params': {'n_estimators': 400, 'max_depth': 10}, 'scores': [0.09348936039289843, 0.6358815990624541, 0.0313768314709562, 0.15358345749590643, 0.06619276461885275], 'mean_mse': np.float64(0.19610480260821356)}


In [33]:
print("XGBoost...")
best_xgb_uni, xgb_results_uni = ts_cross_val_sklearn(build_xgb, xgb_grid, df_uni, target=target)
print(best_xgb_uni)

XGBoost...
{'params': {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.8}, 'scores': [0.11892226826488443, 0.9938241210847093, 0.02616949946207666, 0.1988149764766656, 0.09641567850550417], 'mean_mse': np.float64(0.286829308758768)}


In [35]:
df_uni = df_uni.drop(columns=["datetime"])

In [36]:
print("LSTM...")
best_lstm_uni, lstm_results_uni = ts_cross_val_lstm(df_uni, target=target, n_splits=3, window=24,
                                                     param_grid=lstm_grid, epochs=5, batch_size=32)

LSTM...


C:\Users\Owner\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\Owner\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\Owner\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\Owner\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` arg

In [37]:
print(best_lstm_uni)

{'params': {'lr': 0.001, 'units': 64}, 'scores': [0.5303492743981245, 0.0886665131020457, 0.0528760414324986], 'mean_mse': np.float64(0.22396394297755626)}


In [39]:
# ----------------------------
# 5. Run CV for MULTIVARIATE
# ----------------------------
print("=== MULTIVARIATE CV ===")

print("Linear Regression...")
best_lr_multi, lr_results_multi = ts_cross_val_sklearn(build_linear_regression, lr_grid, df_multi, target=target)
print(best_lr_multi)

print("Random Forest...")
best_rf_multi, rf_results_multi = ts_cross_val_sklearn(build_random_forest, rf_grid, df_multi, target=target)
print(best_rf_multi)

print("XGBoost...")
best_xgb_multi, xgb_results_multi = ts_cross_val_sklearn(build_xgb, xgb_grid, df_multi, target=target)
print(best_xgb_multi)

df_multi = df_multi.drop(columns=["datetime"])
print("LSTM...")
best_lstm_multi, lstm_results_multi = ts_cross_val_lstm(df_multi, target=target, n_splits=3, window=24,
                                                        param_grid=lstm_grid, epochs=5, batch_size=32)
print(best_lstm_multi)


=== MULTIVARIATE CV ===
Linear Regression...
{'params': {}, 'scores': [0.15943666412209406, 0.027566724544333378, 0.02566903359639244, 0.1323389741465395, 0.024068139011415027], 'mean_mse': np.float64(0.07381590708415489)}
Random Forest...
{'params': {'n_estimators': 300, 'max_depth': None}, 'scores': [0.6400492889189453, 0.02329422584200553, 0.03177627858727318, 0.17573560550966144, 0.04310685124825806], 'mean_mse': np.float64(0.18279245002122874)}
XGBoost...
{'params': {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.9}, 'scores': [0.4872828994114175, 0.02490052232584568, 0.031351878169950914, 0.24173789869005768, 0.05745760219577986], 'mean_mse': np.float64(0.16854616015861032)}
LSTM...


C:\Users\Owner\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\Owner\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\Owner\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\Owner\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` arg

{'params': {'lr': 0.001, 'units': 64}, 'scores': [0.472706988726927, 0.19823945414169983, 1.169742430938934], 'mean_mse': np.float64(0.6135629579358536)}


In [48]:
best_lr_multi_params = best_lr_multi["params"]
best_rf_multi_params = best_rf_multi["params"]
best_xgb_multi_params = best_xgb_multi["params"]
best_lstm_multi_params = best_lstm_multi["params"]

best_lr_uni_params = best_lr_uni["params"]
best_rf_uni_params = best_rf_uni["params"]
best_xgb_uni_params = best_xgb_uni["params"]
best_lstm_uni_params = best_lstm_uni["params"]

In [52]:
best_params_multi = {
    "linear_regression": best_lr_multi["params"],
    "random_forest": best_rf_multi["params"],
    "xgboost": best_xgb_multi["params"],
    "lstm": best_lstm_multi["params"]
}

best_params_uni = {
    "linear_regression": best_lr_uni["params"],
    "random_forest": best_rf_uni["params"],
    "xgboost": best_xgb_uni["params"],
    "lstm": best_lstm_uni["params"]
}

with open("best_params_multi.json", "w") as f:
    json.dump(best_params_multi, f, indent=4)

with open("best_params_uni.json", "w") as f:
    json.dump(best_params_uni, f, indent=4)

In [55]:
import joblib

joblib.dump(best_lr_multi, "best_lr_multi.pkl")
joblib.dump(best_rf_multi, "best_rf_multi.pkl")
joblib.dump(best_xgb_multi, "best_xgb_multi.pkl")
joblib.dump(best_lstm_multi, "best_lstm_multi.pkl")

joblib.dump(best_lr_uni, "best_lr_uni.pkl")
joblib.dump(best_rf_uni, "best_rf_uni.pkl")
joblib.dump(best_xgb_uni, "best_xgb_uni.pkl")
joblib.dump(best_lstm_uni, "best_lstm_uni.pkl")

['best_lstm_uni.pkl']